# Phase 3: Controlled Architecture Ablation on Google Colab (NVIDIA CUDA)

**Project:** High-Energy Physics (HEP) Anomaly Detection using Graph Neural Networks on the LHC Olympics 2020 (LHCO2020) Dataset.  
**Scientific Question:** Isolate the individual contributions of:
1. **Graph representation** (Relational $\Delta R < 0.4$ topology vs flat leading-particle MLP)
2. **Physics-informed MET objective** (Conservation constraint head with $\lambda_{\text{MET}} = 0.1$)
3. **Adversarial mass decorrelation** (Gradient Reversal Layer regressing invariant mass with $\lambda_{\text{adv}} = 0.05$)

---

### Models in this Controlled Ablation Campaign
| Model | Family | Architecture | Loss Formulation |
| :--- | :--- | :--- | :--- |
| `baseline_mlp` | `architecture` | `BaselineMLP` (64 &rarr; 32 &rarr; 8 &rarr; 32 &rarr; 64) | $\mathcal{L} = \mathcal{L}_{\text{recon}}$ |
| `gat_standalone` | `architecture` | `NodeGNN` (4-head GAT &rarr; 1-head GAT &rarr; Recon) | $\mathcal{L} = \mathcal{L}_{\text{recon}}$ |
| `gat_met_production` | `architecture` | `NodeGNN` + Global MET head | $\mathcal{L} = \mathcal{L}_{\text{recon}} + 0.1 \times \mathcal{L}_{\text{MET}}$ |
| `gat_met_decorrelation` | `architecture` | `AdversarialNodeGNN` + MET + GRL mass head | $\mathcal{L} = \mathcal{L}_{\text{recon}} + 0.1 \times \mathcal{L}_{\text{MET}} + 0.05 \times \mathcal{L}_{\text{adv}}$ |

---
> **Notice:** Before running, ensure your Colab runtime is set to GPU:  
> `Runtime` &rarr; `Change runtime type` &rarr; Select `T4 GPU` (or `L4`/`A100` if Colab Pro is available).

## 1. Clone / Pull Repository & Set Working Directory

In [ ]:
import os

# Clone repository if running in fresh Colab session, otherwise pull latest commits
if not os.path.exists("anomaly-detector-gnn"):
    !git clone https://github.com/KamitKoul/anomaly-detector-gnn.git
    %cd anomaly-detector-gnn
else:
    %cd anomaly-detector-gnn
    !git pull origin main

!git log -n 3 --oneline

## 2. Mount Google Drive & Link Datasets

The LHCO datasets (~2.5 GB background, ~2.6 GB mixed Zenodo signal) should reside in your Google Drive.
We create symbolic links into `data/` so no large files are copied to Colab local disk.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Specify path to your Drive folder containing the HDF5 datasets
# Change this path if your Drive folder name is different:
DRIVE_DATA_DIR = "/content/drive/MyDrive/lhco_data"

!mkdir -p data

bg_src = os.path.join(DRIVE_DATA_DIR, "events_LHCO2020_backgroundMC_Pythia.h5")
sig_src = os.path.join(DRIVE_DATA_DIR, "W_prime_signal.h5")

if os.path.exists(bg_src) and os.path.exists(sig_src):
    !ln -sf "{bg_src}" data/events_LHCO2020_backgroundMC_Pythia.h5
    !ln -sf "{sig_src}" data/W_prime_signal.h5
    print("[SUCCESS] Datasets symlinked from Google Drive:")
    !ls -lh data/
else:
    print(f"[WARNING] Datasets not found at: {DRIVE_DATA_DIR}")
    print("Please verify your Drive folder path and file names:")
    print("  1. events_LHCO2020_backgroundMC_Pythia.h5")
    print("  2. W_prime_signal.h5")

## 3. Automated Environment Verification & Diagnostics

Executes `scripts/setup_colab.sh` to install dependencies, verify CUDA, run unit tests, and check dry-runs.

In [ ]:
!bash scripts/setup_colab.sh

## 4. Run Isolated GAT CUDA Smoke Test

Validates forward pass, loss computation, backward pass, and CUDA memory allocation on a 20-event subset before launching full training.

In [ ]:
!python src/train_experiment.py \
    --config configs/architecture/gat_met_production.json \
    --limit 20 \
    --epochs 1 \
    --smoke-test \
    --output-dir experiments/smoke_test/colab_smoke_test

!rm -rf experiments/smoke_test/colab_smoke_test
print("[VERIFIED] GAT model executes cleanly on NVIDIA CUDA GPU.")

## 5. Phase 3: Train the Controlled Architecture Ablation Variants

Scientific Conditions (Preserved strictly across all experiments):
- `seed`: 42
- `train_limit`: 5,000 background events
- `batch_size`: 32
- `learning_rate`: 0.001 (Adam)
- Outputs saved to `experiments/architecture/<experiment_name>/`

### 5.1 Experiment 1: Baseline MLP Autoencoder
*(Skip if already trained locally on MPS at `experiments/architecture/baseline_mlp/checkpoint.pt`)*

In [ ]:
if os.path.exists("experiments/architecture/baseline_mlp/checkpoint.pt"):
    print("[INFO] Baseline MLP checkpoint already exists. Skipping training.")
else:
    !python src/train_experiment.py --config configs/architecture/baseline_mlp.json

### 5.2 Experiment 2: Standalone GAT Autoencoder
- Model: `NodeGNN` (conv1 4-head GAT &rarr; conv2 1-head GAT &rarr; recon head)
- Loss: Pure reconstruction MSE ($\lambda_{\text{MET}} = 0.0$, $\lambda_{\text{adv}} = 0.0$)
- Epochs: 20

In [ ]:
!python src/train_experiment.py --config configs/architecture/gat_standalone.json

### 5.3 Experiment 3: Production GAT + MET Objective (Reference Baseline)
- Model: `NodeGNN` (Multi-Task GAT autoencoder + global MET prediction head)
- Loss: $\mathcal{L} = \mathcal{L}_{\text{recon}} + 0.1 \times \mathcal{L}_{\text{MET}}$
- Epochs: 20

In [ ]:
!python src/train_experiment.py --config configs/architecture/gat_met_production.json

### 5.4 Experiment 4: GAT + MET + Adversarial Mass Decorrelation
- Model: `AdversarialNodeGNN` (Multi-Task GAT + MET + GRL mass adversary)
- Loss: $\mathcal{L} = \mathcal{L}_{\text{recon}} + 0.1 \times \mathcal{L}_{\text{MET}} + 0.05 \times \mathcal{L}_{\text{adv}}$
- Epochs: 15

In [ ]:
!python src/train_experiment.py --config configs/architecture/gat_met_decorrelation.json

## 6. Single-Experiment Evaluation

Scores 500 background events and 500 filtered genuine signal events (`truth == 1`) for each trained model, computing:
- ROC AUC
- Background Rejection ($1 / \text{FPR}$) at 30% and 50% signal efficiency
- Statistical Separation Ratio ($\mu_{\text{sig}} / \mu_{\text{bg}}$)
- Separation Significance ($\sigma$)
- Generates `metrics.json` and `predictions.npz` in each experiment directory.

In [ ]:
experiments = [
    "experiments/architecture/baseline_mlp",
    "experiments/architecture/gat_standalone",
    "experiments/architecture/gat_met_production",
    "experiments/architecture/gat_met_decorrelation",
]

for exp in experiments:
    print(f"\nEvaluating {exp}...")
    !python evaluation/evaluate_single.py --experiment "{exp}"

## 7. Comparative Analysis & Summary Aggregation

Aggregates the evaluation metrics from all 4 models and generates a comparison table in `comparisons/architecture/summary.md`.

In [ ]:
!python evaluation/compare_experiments.py --family architecture

print("\n--- Comparative Results Summary ---")
!cat comparisons/architecture/summary.md

## 8. Backup Results to Google Drive

Safely persists the training logs, metrics, predictions, and checkpoints back to Google Drive for offline inspection in Mac VS Code.

In [ ]:
BACKUP_DIR = "/content/drive/MyDrive/anomaly_detector_experiments"
!mkdir -p "{BACKUP_DIR}"

!cp -r experiments/architecture "{BACKUP_DIR}/"
!cp -r comparisons/architecture "{BACKUP_DIR}/"

print(f"[SUCCESS] Experiment results successfully backed up to: {BACKUP_DIR}")
!ls -lh "{BACKUP_DIR}/architecture"